Load raw data + set up a transformation log

In [2]:
import os
print(os.listdir("D:/asg-airlines-pipeline/data/raw"))

['asg_airlines_raw.xlsx']


In [3]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
"D:/asg-airlines-pipeline/data/raw/asg_airlines_raw.xlsx"
RAW_PATH = 

flights    = pd.read_excel(RAW_PATH, sheet_name="flights")
payments   = pd.read_excel(RAW_PATH, sheet_name="payments")
bookings   = pd.read_excel(RAW_PATH, sheet_name="bookings")
passengers = pd.read_excel(RAW_PATH, sheet_name="passengers")

transform_log = []

def log_step(table, action, before, after):
    transform_log.append({
        "table": table, "action": action,
        "rows_before": before, "rows_after": after,
        "rows_affected": before - after if before >= after else "n/a (values changed, not dropped)"
    })

In [5]:
before = len(flights)
flights = flights.drop_duplicates()
log_step("flights", "drop full duplicates", before, len(flights))

In [ ]:
dupe_ids = flights[flights.duplicated("flight_id", keep=False)].sort_values("flight_id")
print(dupe_ids)  
flights["flight_id"] = flights["flight_id"] + "_" + flights.groupby("flight_id").cumcount().astype(str).replace("0","")
flights["flight_id"] = flights["flight_id"].str.rstrip("_")

    flight_id  airline source destination          departure_time  \
253     6F250  UNKNOWN    DEL         BLR 2026-04-20 03:26:41.701   
270     6F250  UNKNOWN    CCU         BLR 2026-04-20 02:23:41.702   

               arrival_time  duration  
253 2026-04-20 07:30:41.701  04:04:00  
270 2026-04-20 02:56:41.702  00:33:00  


In [ ]:
flights["is_corrupted_time"] = flights["arrival_time"] < flights["departure_time"]
print("Corrupted time records flagged:", flights["is_corrupted_time"].sum())


Corrupted time records flagged: 1


In [8]:
flights["airline"] = flights["airline"].fillna("UNKNOWN")
flights["airline"] = flights["airline"].str.strip().str.title().replace({"Unknown": "UNKNOWN"})

In [9]:
flights["duration_calc"] = flights["arrival_time"] - flights["departure_time"]
flights["duration_minutes"] = flights["duration_calc"].dt.total_seconds() / 60
flights.loc[flights["is_corrupted_time"], "duration_minutes"] = np.nan  # exclude bad rows
flights["is_overnight"] = flights["arrival_time"].dt.date > flights["departure_time"].dt.date

In [10]:
before_invalid = (payments["amount"].astype(str) == "INVALID").sum()
payments["amount"] = pd.to_numeric(payments["amount"], errors="coerce")
print(f"Converted {before_invalid} 'INVALID' string values to NaN")

Converted 30 'INVALID' string values to NaN


In [ ]:
payments["amount_is_missing"] = payments["amount"].isna()
payments["amount_median_imputed"] = payments["amount"].fillna(payments["amount"].median())

In [12]:
bookings["status"] = bookings["status"].replace("INVALID", np.nan)
bookings["status"] = bookings["status"].fillna("UNKNOWN")
bookings["status"] = bookings["status"].str.strip().str.upper()

In [ ]:
orphan_bookings = bookings[~bookings["flight_id"].isin(flights["flight_id"])]
print("Orphan bookings after flight_id cleanup:", len(orphan_bookings))


Orphan bookings after flight_id cleanup: 0


In [14]:
passengers["last_name"] = passengers["last_name"].fillna("UNKNOWN")

In [15]:
invalid_age = passengers[(passengers["age"] < 0) | (passengers["age"] > 120)]
print("Invalid ages:", len(invalid_age))

Invalid ages: 0


In [17]:
flights.to_pickle("../data/processed/flights_clean.pkl")
payments.to_pickle("../data/processed/payments_clean.pkl")
bookings.to_pickle("../data/processed/bookings_clean.pkl")
passengers.to_pickle("../data/processed/passengers_clean.pkl")

print("Saved cleaned tables:")
print(f"  flights: {flights.shape}")
print(f"  payments: {payments.shape}")
print(f"  bookings: {bookings.shape}")
print(f"  passengers: {passengers.shape}")

Saved cleaned tables:
  flights: (1005, 11)
  payments: (1000, 6)
  bookings: (1000, 9)
  passengers: (1039, 9)


In [19]:
%pip install pyarrow

   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   - -------------------------------------- 0.8/27.9 MB 7.5 MB/s eta 0:00:04
   --- ------------------------------------ 2.6/27.9 MB 8.1 MB/s eta 0:00:04
   ----- ---------------------------------- 3.7/27.9 MB 7.6 MB/s eta 0:00:04
   ------ --------------------------------- 4.7/27.9 MB 6.6 MB/s eta 0:00:04
   -------- ------------------------------- 6.0/27.9 MB 6.4 MB/s eta 0:00:04
   ---------- ----------------------------- 7.1/27.9 MB 6.3 MB/s eta 0:00:04
   ----------- ---------------------------- 8.1/27.9 MB 5.9 MB/s eta 0:00:04
   ------------- -------------------------- 9.2/27.9 MB 6.0 MB/s eta 0:00:04
   -------------- ------------------------- 10.2/27.9 MB 5.8 MB/s eta 0:00:04
   ---------------- ----------------------- 11.3/27.9 MB 5.6 MB/s eta 0:00:03
   ----------------- ---------------------- 12.3/27.9 MB 5.6 MB/s eta 0:00:03
   ------------------- -------------------- 13.4/27.9 MB 5.5 MB/s eta 0:00:03
  


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [21]:
%pip install pandas numpy openpyxl pyarrow sqlalchemy matplotlib

  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.3.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.2 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 5.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.3 MB 9.8 MB/s eta 0:00:01
   ------------ --------------------------- 2.9/9.3 MB 7.7 MB/s eta 0:00:01
   ------------------- -------------------- 4.5/9.3 MB 7.6 MB/s eta 0:00:01
   ------------------------- -------------- 6.0/9.3 MB 7.7 MB/s eta 0:00:01
   ------------------------------- -------- 7.3/9.3 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------  9.2/9.3 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
